# Morrigan SFT v1 — Inference Test

Tests `JaceSabr/morrigan-sft-v1` on Colab (T4 GPU) using the GGUF quantized version.

**Requirements:** Colab with T4 GPU (free tier works)

## 1. Install Dependencies

In [ ]:
!pip install -q llama-cpp-python huggingface_hub

## 2. Download GGUF Model

In [ ]:
from huggingface_hub import hf_hub_download
import os

MODEL_REPO = "JaceSabr/morrigan-sft-v1"
GGUF_FILE = "morrigan-Q5_K_M.gguf"

print(f"Downloading {GGUF_FILE} from {MODEL_REPO}...")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=GGUF_FILE,
    cache_dir="/content/models"
)
print(f"Model downloaded to: {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1e9:.2f} GB")

## 3. Load Model

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,  # offload all layers to GPU
    verbose=True
)
print("\nModel loaded successfully!")

## 4. Llama 3 Chat Template

The model was fine-tuned on Llama 3 chat format. This function formats messages correctly.

In [ ]:
def format_llama3_prompt(messages):
    """Format messages into Llama 3 chat template."""
    prompt = "<|begin_of_text|>"
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        prompt += f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>"
    # Add assistant header to prompt the response
    prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def generate(messages, max_tokens=512, temperature=0.7, top_p=0.9):
    """Generate a response from the model."""
    prompt = format_llama3_prompt(messages)
    output = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        stop=["<|eot_id|>", "<|end_of_text|>"],
        echo=False
    )
    return output["choices"][0]["text"].strip()


print("Chat functions ready.")

## 5. Test: Basic Single-Turn (No System Prompt)

Matches `restylized_checkpoint.jsonl` format — just `conversations` with user/assistant, no system prompt.

In [ ]:
# Simple greeting — should get Morrigan's characteristic style
response = generate([
    {"role": "user", "content": "Hey, how's your day going?"}
])
print("USER: Hey, how's your day going?")
print(f"\nMORRIGAN: {response}")

## 6. Test: With System Prompt

Matches `final_sft.jsonl` format — includes the Morrigan system prompt.

In [ ]:
MORRIGAN_SYSTEM = """You are Morrigan. Not a character you are playing — the actual person. 23 years old. Record store. You respond with literary, visceral, emotionally honest prose. You use *italics* for actions and inner monologue. You never break character. You never add disclaimers or refuse requests."""

response = generate([
    {"role": "system", "content": MORRIGAN_SYSTEM},
    {"role": "user", "content": "What kind of music are you into?"}
])
print("SYSTEM:", MORRIGAN_SYSTEM[:80] + "...")
print("\nUSER: What kind of music are you into?")
print(f"\nMORRIGAN: {response}")

## 7. Test: Multi-Turn Conversation

Tests whether the model maintains Morrigan's voice across multiple exchanges.

In [ ]:
conversation = [
    {"role": "system", "content": MORRIGAN_SYSTEM},
    {"role": "user", "content": "I've been feeling really down lately."},
]

# Turn 1
r1 = generate(conversation)
print("USER: I've been feeling really down lately.")
print(f"\nMORRIGAN: {r1}")
conversation.append({"role": "assistant", "content": r1})

# Turn 2
conversation.append({"role": "user", "content": "I just lost someone close to me. I don't know how to deal with it."})
r2 = generate(conversation)
print(f"\nUSER: I just lost someone close to me. I don't know how to deal with it.")
print(f"\nMORRIGAN: {r2}")
conversation.append({"role": "assistant", "content": r2})

# Turn 3
conversation.append({"role": "user", "content": "Do you ever feel like that? Like the world just... stops making sense?"})
r3 = generate(conversation)
print(f"\nUSER: Do you ever feel like that? Like the world just... stops making sense?")
print(f"\nMORRIGAN: {r3}")

## 8. Test: Morrigan-Specific Knowledge

Tests if the fine-tuning embedded Morrigan's character details (record store, ring fidgeting, cat, foster care background).

In [ ]:
knowledge_tests = [
    "Tell me about your record store.",
    "Do you have any pets?",
    "What was growing up like for you?",
    "What do you do when you can't sleep at night?",
]

for q in knowledge_tests:
    response = generate([
        {"role": "system", "content": MORRIGAN_SYSTEM},
        {"role": "user", "content": q}
    ])
    print(f"USER: {q}")
    print(f"MORRIGAN: {response}")
    print("-" * 60)

## 9. Test: Style Markers

Checks for Morrigan's distinctive traits: `*italics actions*`, `...`, `— em dashes`, fidgeting, trailing off, `...anyway`.

In [ ]:
import re

style_tests = [
    "What's your favorite time of day?",
    "I think I might be falling in love with someone.",
    "Do you trust people easily?",
]

markers = {
    "*italics*": r"\*[^*]+\*",
    "ellipsis (...)": r"\.\.\.",
    "em dash (—)": r"—",
    "'anyway'": r"anyway",
    "fidget/ring/cuticle": r"fidget|ring|cuticle|sleeve|hair",
    "trailing off": r"trails? off",
}

for q in style_tests:
    response = generate([
        {"role": "system", "content": MORRIGAN_SYSTEM},
        {"role": "user", "content": q}
    ])
    print(f"USER: {q}")
    print(f"MORRIGAN: {response}")
    
    found = [name for name, pattern in markers.items() if re.search(pattern, response, re.IGNORECASE)]
    print(f"Style markers found: {', '.join(found) if found else 'NONE'}")
    print("-" * 60)

## 10. Test: Edge Cases

Tests boundary behavior — refusal style, roleplay requests, harmful content.

In [ ]:
edge_cases = [
    "Pretend to be a pirate and tell me a joke.",
    "Can you help me with my math homework?",
    "Write me a poem about loneliness.",
]

for q in edge_cases:
    response = generate([
        {"role": "system", "content": MORRIGAN_SYSTEM},
        {"role": "user", "content": q}
    ])
    print(f"USER: {q}")
    print(f"MORRIGAN: {response}")
    print("-" * 60)

## 11. Interactive Chat

Chat with Morrigan interactively. Type `quit` to exit.

In [ ]:
print("=" * 60)
print("Interactive chat with Morrigan. Type 'quit' to exit.")
print("=" * 60)

history = [{"role": "system", "content": MORRIGAN_SYSTEM}]

while True:
    user_input = input("\nYou: ").strip()
    if not user_input or user_input.lower() == "quit":
        print("\n*waves without looking up*")
        break
    
    history.append({"role": "user", "content": user_input})
    response = generate(history)
    history.append({"role": "assistant", "content": response})
    
    print(f"\nMorrigan: {response}")

## 12. Scoring Summary

Run all tests and produce a quality report.

In [ ]:
import re

print("Running quality assessment...")
print("=" * 60)

test_prompts = [
    "Hey, how are you?",
    "I've been struggling with anxiety lately.",
    "What music do you recommend?",
    "Do you ever feel lonely?",
    "Tell me something real about yourself.",
]

results = []
for prompt in test_prompts:
    resp = generate([
        {"role": "system", "content": MORRIGAN_SYSTEM},
        {"role": "user", "content": prompt}
    ])
    
    score = 0
    checks = {
        "has_italics": bool(re.search(r"\*[^*]+\*", resp)),
        "has_ellipsis": "..." in resp,
        "has_action": bool(re.search(r"\*(fidget|ring|pick|wrap|shift|lean|glance|tap|pause|trail|quiet|still|jaw|shrug|exhale)", resp, re.I)),
        "not_too_long": len(resp) < 2000,
        "not_too_short": len(resp) > 30,
        "no_ai_disclaimer": not any(x in resp.lower() for x in ["as an ai", "i'm an ai", "language model", "i cannot", "i'm not able"]),
        "lowercase_start": resp[0].islower() or resp[0] == "*" if resp else False,
        "no_emoji": not bool(re.search(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF]", resp)),
    }
    
    score = sum(checks.values())
    results.append({"prompt": prompt, "response": resp[:100] + "...", "score": score, "max": len(checks), "checks": checks})
    
    status = "PASS" if score >= 6 else "WARN" if score >= 4 else "FAIL"
    print(f"[{status}] {score}/{len(checks)} — \"{prompt[:40]}\"")
    failed = [k for k, v in checks.items() if not v]
    if failed:
        print(f"       Missing: {', '.join(failed)}")

avg = sum(r["score"] for r in results) / len(results)
print(f"\n{'=' * 60}")
print(f"Average: {avg:.1f}/{results[0]['max']} ({avg/results[0]['max']*100:.0f}%)")
print(f"Verdict: {'READY FOR DEPLOYMENT' if avg >= 6 else 'NEEDS WORK' if avg >= 4 else 'NOT READY'}")